# Assignment 4 — Context distraction

Связь с Assignment 3: Objective 4, relevance and context use. Новые результаты появятся только после реального запуска. Выполняйте ячейки по порядку. В Colab выберите Runtime → Change runtime type → GPU. Не запускайте final до загрузки протокола в Moodle.

## 1 Клонируйте проект из GitHub
Ячейка создаёт /content/RMT_Assignment4. Если папка уже существует, используется существующая копия. Для обновления сначала сохраните свои результаты.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, sys, os
project = Path('/content/RMT_Assignment4')
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/despi4/RMT_Assignment4.git', str(project)], check=True)
assert (project / 'config/experiment.json').is_file(), 'Нужна папка проекта из GitHub'
os.chdir(project)
print('Project:', project)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


## 2 Установите зависимости и проверьте код
Смена пакетов после freeze запрещена. Сначала установка и dev pilot, затем Moodle upload и freeze.

In [ ]:
%pip install -r requirements.txt
import subprocess, sys
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([sys.executable, "scripts/prepare.py"], check=True)

## 3 Pilot на dev данных
63 пробных вызова на 3 development фактах. Проверьте формат ответов, память и время. Это не финальные результаты. При необходимости настройте код только по dev до freeze. Любые изменения должны попасть в окончательный протокол.

In [ ]:
subprocess.run([sys.executable, "scripts/run.py", "--phase", "pilot", "--limit-facts", "3"], check=True)

## 4 Скачайте данные для заполнения Appendix A
Вставьте настоящий model revision и hashes в форму Appendix A из задания. Текст протокола находится в docs/PROTOCOL.md. Добавьте имена, оборудование и разрешение преподавателя. Загрузите заполненную форму в Moodle ДО final. Сохраните реальный receipt. Сначала убедитесь, что нет незафиксированных изменений после pilot.

In [ ]:
import json
print(Path('config/model_lock.json').read_text())
print(Path('data/manifest.json').read_text())
print(Path('config/experiment.json').read_text())
files.download('docs/PROTOCOL.md')

## 5 Запишите настоящую загрузку Moodle
Эта ячейка требует действий человека. Не подставляйте вымышленную дату или картинку. Имя и дата заполняются вручную. Receipt может содержать персональные данные — не публикуйте оригинал в открытом GitHub.

In [ ]:
receipt_upload = files.upload()
receipt_file = next(iter(receipt_upload))
UPLOADED_AT = ''  # ISO timestamp с timezone из настоящей загрузки Moodle
DECLARANT = ''    # ваше имя
assert UPLOADED_AT and DECLARANT, 'Сначала заполните реальную дату загрузки и имя'
import subprocess, sys
subprocess.run([sys.executable, 'scripts/freeze.py', '--uploaded-at', UPLOADED_AT, '--evidence', receipt_file, '--declarant', DECLARANT], check=True)

## 6 Финальный запуск
2520 вызовов. Не меняйте данные, настройки, prompt, зависимости. Не делайте вывод по нескольким первым ответам. После прерывания команда продолжит тот же лог при неизменной конфигурации; ошибки сохраняются и требуют оформленного deviation.

In [ ]:
subprocess.run([sys.executable, 'scripts/run.py', '--phase', 'final'], check=True)

## 7 Анализ и графики
120 независимых fact clusters; 3 context seeds. UNKNOWN в no-context контроле не означает знание города.

In [ ]:
subprocess.run([sys.executable, 'scripts/analyze.py'], check=True)
import pandas as pd
display(pd.read_csv('results/analysis/main_results.csv'))
display(pd.read_csv('results/analysis/paired_comparisons.csv'))
from IPython.display import SVG, display
for filename in ['figure_1_accuracy.svg','figure_2_noise.svg','figure_3_position.svg']:
    display(SVG(filename='results/analysis/'+filename))

## 8 Человеческий анализ ошибок
Скачайте error_review.csv. Два участника независимо заполняют rater_1/rater_2, затем final_category и explanation. Разрешённые категории — в docs/PROTOCOL.md. Не подменяйте файл случайными метками.

In [ ]:
files.download('results/analysis/error_review.csv')

In [ ]:
review_upload = files.upload()
review_name = next(iter(review_upload))
Path('results/analysis/error_review.csv').write_bytes(review_upload[review_name])
RATER_1 = ''  # actual first human rater
RATER_2 = ''  # actual second human rater
assert RATER_1 and RATER_2
subprocess.run([sys.executable, 'scripts/review_errors.py', '--rater-1-name', RATER_1, '--rater-2-name', RATER_2], check=True)

## 9 Резервная копия и сдача
Скачайте проект, оформите Word-отчёт настоящими результатами, выполните независимое воспроизведение и Appendix C. Добавьте минимум три источника из вашего Assignment 1; его наличие здесь не предполагается. Создайте GitHub/GitLab release assignment-4. Финальная защита индивидуальная.

In [ ]:
import shutil
backup = shutil.make_archive('/content/Assignment4_with_actual_results', 'zip', root_dir=project.parent, base_dir=project.name)
files.download(backup)

## Проверка всех критериев
Запустите аудит. Статус CODE READY не означает, что исследование уже выполнено. Для независимого воспроизведения используйте отдельный режим из README.

In [ ]:
subprocess.run([sys.executable, 'scripts/submission_check.py'], check=True)
